# 02 — Nested evaluation (never fits)

`exp5_touch_ens3` and the pooled `exp45_touch_ens6` on every S1 label set (provider, `L_track(1 s)`, `L_ctrl`, `L_track(3 s)`) and both views; single seeds on provider and `L_ctrl`, both views. Temperature and decoder are re-selected on the six inner validation matches against each label set; the held-out match is scored once. `exp3_ref_ens3` / `exp2_aug` (exp_3) and `exp4_touch_ens3` (exp_4) are read from stored results. Then argmax diagnostics, recall of possessions < 2 s and the NLL generalisation gap.

In [1]:
from pathlib import Path
import os, sys, json, time
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp5_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_5')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp5_common as C
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp5_protocol as P, exp5_eval as E, exp5_diagnostics as G
for arm in P.ARMS:
    assert not P.missing_jobs(arm), f'{arm}: jobs missing; run 01_fits first'

PREREG verified dc28ebfcacfa2676 | frozen 2026-11-10T09:06:10+00:00


In [2]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
tasks = E.tasks()
t0 = time.time()
with ProcessPoolExecutor(max_workers=2, mp_context=multiprocessing.get_context('fork')) as pool:  # RAM ~3 GB free
    for variant, view, label_set, seconds, tag in pool.map(E.evaluate_and_save, tasks):
        frame = pd.DataFrame(E.read_result(variant, view, label_set)['rows'])
        print(f'{label_set:8s} {view:12s} {variant:17s} spellF1 {frame.spell_f1.mean():.4f}  max|cnt| {frame.count_error_ratio.abs().max():.3f}  gamma {sorted(set(frame.gamma))}  ({tag} {seconds:.0f}s)', flush=True)
print(f'{len(tasks)} evaluations, {time.time() - t0:.0f}s')

provider original     exp5_touch_ens3   spellF1 0.6760  max|cnt| 0.317  gamma [0.0]  (computed 97s)


provider target_like  exp5_touch_ens3   spellF1 0.6743  max|cnt| 0.329  gamma [0.0]  (computed 97s)


track_1  original     exp5_touch_ens3   spellF1 0.7149  max|cnt| 0.198  gamma [0.0]  (computed 87s)


track_1  target_like  exp5_touch_ens3   spellF1 0.7105  max|cnt| 0.216  gamma [0.0, 0.5]  (computed 87s)


ctrl     original     exp5_touch_ens3   spellF1 0.7452  max|cnt| 0.195  gamma [0.0]  (computed 80s)


ctrl     target_like  exp5_touch_ens3   spellF1 0.7400  max|cnt| 0.187  gamma [0.0]  (computed 79s)


... (19 outputs left out)

provider target_like  exp5_touch_s2     spellF1 0.6593  max|cnt| 0.342  gamma [0.0]  (computed 94s)


ctrl     original     exp5_touch_s2     spellF1 0.7324  max|cnt| 0.171  gamma [0.0, 0.5, 1.0]  (computed 81s)


ctrl     target_like  exp5_touch_s2     spellF1 0.7291  max|cnt| 0.200  gamma [0.0]  (computed 81s)


28 evaluations, 1192s


## Diagnostics: argmax (no decoder) and recall of true possessions < 2 s (provider labels)

In [3]:
cache = {}
argmax = pd.DataFrame([r for v in (C.PRIMARY, C.REFERENCE) for ls in C.DECISION_LABELS for view in C.VIEWS
                       for r in G.argmax_rows(v, view, ls, cache)])
argmax.to_csv(C.RESULTS / 'argmax_rows.csv', index=False)
print(argmax.groupby(['label_set', 'view', 'variant'])[['spell_f1', 'count_error_ratio']].mean().round(4).to_string())
recall_variants = (C.PRIMARY, C.REFERENCE, 'exp4_touch_ens3', 'exp45_touch_ens6', 'exp2_aug')
recall = pd.DataFrame([r for v in recall_variants for view in C.VIEWS for r in G.short_recall_rows(v, view, cache)])
recall.to_csv(C.RESULTS / 'short_recall.csv', index=False)
print(recall.groupby(['view', 'variant'])[['recall_short_decoded', 'recall_short_argmax', 'recall_all_decoded']].mean().round(4).to_string())

                                              spell_f1  count_error_ratio
label_set view        variant                                            
ctrl      original    exp3_ref_ens3_argmax      0.6355             0.3973
                      exp5_touch_ens3_argmax    0.6452             0.3771
          target_like exp3_ref_ens3_argmax      0.6312             0.4016
                      exp5_touch_ens3_argmax    0.6409             0.3791
provider  original    exp3_ref_ens3_argmax      0.6475            -0.0716
                      exp5_touch_ens3_argmax    0.6513            -0.0862
          target_like exp3_ref_ens3_argmax      0.6426            -0.0687
                      exp5_touch_ens3_argmax    0.6505            -0.0848


                              recall_short_decoded  recall_short_argmax  recall_all_decoded
view        variant                                                                        
original    exp2_aug                        0.2887               0.3263              0.6072
            exp3_ref_ens3                   0.3017               0.3481              0.6096
            exp45_touch_ens6                0.3099               0.3387              0.6174
            exp4_touch_ens3                 0.2976               0.3313              0.6177
            exp5_touch_ens3                 0.3184               0.3480              0.6157
target_like exp2_aug                        0.2869               0.3225              0.6069
            exp3_ref_ens3                   0.3006               0.3424              0.6095
            exp45_touch_ens6                0.3135               0.3397              0.6149
            exp4_touch_ens3                 0.3016               0.3275         

## Generalisation gap (provider labels, raw p, T = 1)
Training NLL of every exp_5 fit (logged at fit time) and of the exp_2 recipe (exp_4's `ref_train_nll.csv`, read in place) against the held-out NLL of their outer and pair predictions.

In [4]:
t0 = time.time()
train = pd.concat([pd.DataFrame([r for arm in P.ARMS for r in P.train_nll_rows(arm)]), G.reference_train_nll()],
                  ignore_index=True)
heldout = pd.DataFrame([r for arm in (*P.ARMS, *E.EXP2_ARMS) for r in G.heldout_nll_rows(arm, cache)])
train.to_csv(C.RESULTS / 'train_nll.csv', index=False)
heldout.to_csv(C.RESULTS / 'heldout_nll.csv', index=False)
gap = G.gap_table(train, heldout)
gap.to_csv(C.RESULTS / 'gap_table.csv', index=False)
print(gap.round(4).to_string(index=False))
print(f'{time.time() - t0:.0f}s')

          arm  train_nll  heldout_nll_original  heldout_nll_target_like  gap_original  gap_target_like      family  gap_growth_original  gap_growth_target_like  level flag_gap_growth_gt_25pct
     exp2_aug     0.1165                0.1888                   0.1919        0.0722           0.0754 exp2_recipe               0.0048                  0.0082    arm                      NaN
  exp2_aug_s1     0.1161                0.1884                   0.1916        0.0723           0.0754 exp2_recipe               0.0053                  0.0095    arm                      NaN
  exp2_aug_s2     0.1169                0.1881                   0.1903        0.0712           0.0734 exp2_recipe              -0.0102                 -0.0176    arm                      NaN
   exp5_touch     0.1221                0.1756                   0.1814        0.0535           0.0593  exp5_touch              -0.2560                 -0.2064    arm                      NaN
exp5_touch_s1     0.1235                